# 📊 US Market Screener — Google Colab
**Screens all NASDAQ + NYSE/AMEX common stocks (~6,500) with three quantitative scans:**
- 🔲 **Darvas Box** — technical momentum breakout (batch OHLC, fast)
- 📐 **Piotroski F-Score** — 9-point financial-strength score
- ☕ **Coffee Can** — quality + growth filter (ROE, FCF, low debt, no loss year)

| Phase | Method | Est. Time |
|---|---|---|
| 1 — Darvas | `yf.download()` batch (500 tickers/call) | ~3–5 min |
| 2 — Fundamentals | `ThreadPoolExecutor` (10 workers) | ~35–50 min |

**Output files** (saved to Google Drive or `/content/us_screener_output/`):
```
scan_summary_YYYYMMDD.csv       all stocks + all scan results
darvas_breakouts_YYYYMMDD.csv   BREAKOUT_BUY signals only
strong_piotroski_YYYYMMDD.csv   F-Score >= 7
coffee_can_YYYYMMDD.csv         Coffee Can qualifiers
triple_hits_YYYYMMDD.csv        ★ pass all 3 scans (highest conviction)
screener.db                     SQLite checkpoint (enables --resume)
```

---
**How to use this notebook:**
1. Run **Cell 2** (install) once per Colab session
2. Edit **Cell 3** (configuration) to set your preferences
3. Run **Cell 4** (Drive mount) to persist results across sessions *(optional)*
4. Run **Cell 5** (clear cache) only if you want to start fresh
5. Run **Cells 6–11** to load all functions (Run All works fine)
6. Run **Cell 12** 🚀 to start the screener
7. Run **Cell 13** to view the results summary


In [ ]:
# ── Cell 2: Install dependencies (run once per Colab session) ───────────────
!pip install yfinance pandas tqdm requests --quiet
print('✅ Dependencies ready')


In [ ]:
# ── Cell 3: CONFIGURATION — edit these before running ───────────────────────

EXCHANGE     = 'all'    # 'all' | 'nasdaq' | 'nyse' | 'amex'
LIMIT        = 0        # 0 = all ~6,500 stocks; use 100-200 for a quick test
RESUME       = False    # True = skip symbols already saved in screener.db
DARVAS_ONLY  = False    # True = skip Piotroski + Coffee Can (much faster)
BATCH_SIZE   = 500      # tickers per yf.download() bulk call
WORKERS      = 15       # concurrent threads for fundamentals (Colab allows ~15)
DELAY        = 0.05     # seconds between API calls per worker (rate-limit guard)

USE_DRIVE    = True     # True = save outputs to Google Drive (recommended)
DRIVE_FOLDER = 'us_screener_output'  # subfolder inside MyDrive

print('Configuration loaded')


In [ ]:
# ── Cell 4: Google Drive mount (recommended — keeps results across sessions) ─
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = Path(f'/content/drive/MyDrive/{DRIVE_FOLDER}')
    print(f'✅ Drive mounted — outputs will be saved to {OUTPUT_DIR}')
else:
    OUTPUT_DIR = Path('/content/us_screener_output')
    print(f'ℹ️  Saving locally to {OUTPUT_DIR}  (lost when Colab session ends)')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DB_FILE = OUTPUT_DIR / 'screener.db'
print(f'Database: {DB_FILE}')


In [ ]:
# ── Cell 5: CLEAR CACHE ──────────────────────────────────────────────────────
# Run this cell ONLY if you want to discard all previous results and
# restart the screener from scratch.
# Leave CLEAR_CACHE = False to skip.

CLEAR_CACHE = False   # ← set True then run this cell to reset

if CLEAR_CACHE:
    import shutil

    # 1. Remove SQLite checkpoint (progress + results)
    if DB_FILE.exists():
        DB_FILE.unlink()
        print(f'✅ Deleted checkpoint: {DB_FILE}')
    else:
        print('   No checkpoint found.')

    # 2. Remove cached stock-universe CSV
    universe_cache = OUTPUT_DIR / 'stock_universe.csv'
    if universe_cache.exists():
        universe_cache.unlink()
        print(f'✅ Deleted universe cache: {universe_cache}')

    # 3. Remove all previous CSV outputs for today
    from datetime import datetime
    today = datetime.today().strftime('%Y%m%d')
    for f in OUTPUT_DIR.glob(f'*_{today}.csv'):
        f.unlink()
        print(f'✅ Deleted: {f.name}')

    # 4. Clear yfinance local price cache
    import shutil
    yf_cache = Path.home() / '.cache' / 'py-yfinance'
    if yf_cache.exists():
        shutil.rmtree(yf_cache)
        print(f'✅ Cleared yfinance cache: {yf_cache}')

    print('\n🔄  All caches cleared. Next run starts fresh.')
else:
    print('ℹ️  CLEAR_CACHE = False — no cache cleared.')
    print('    Set CLEAR_CACHE = True and re-run this cell to reset.')


In [ ]:
# ── Cell 6: Imports and constants ────────────────────────────────────────────
import io, json, logging, sqlite3, sys, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

import pandas as pd
import requests
import yfinance as yf
from tqdm.notebook import tqdm   # notebook-friendly progress bar

logging.basicConfig(level=logging.WARNING,
                    format='%(asctime)s %(levelname)-8s %(message)s',
                    datefmt='%H:%M:%S')
log = logging.getLogger('screener')

NASDAQ_URL     = 'https://www.nasdaqtrader.com/dynamic/SymDir/nasdaqlisted.txt'
OTHER_URL      = 'https://www.nasdaqtrader.com/dynamic/SymDir/otherlisted.txt'
DARVAS_CONFIRM = 3   # days a high/low must hold unbroken
PIOTROSKI_STRONG    = 7
COFFEE_CAN_CRITERIA = 6

print('✅ Imports loaded')


In [ ]:
# ── Cell 7: Scan algorithms ──────────────────────────────────────────────────

# ── Darvas Box ───────────────────────────────────────────────────────────────
def _darvas_from_df(df, confirm=DARVAS_CONFIRM):
    """
    Compute Darvas Box signal from a flat OHLC DataFrame.
    The current bar (last row) is EXCLUDED from box formation so a breakdown
    bar cannot pull the box bottom to its own low.
    Returns dict: signal, box_top, box_bottom, current_price, upside_pct.
    """
    if df is None or df.empty or len(df) < confirm + 5:
        return {'signal': 'INSUFFICIENT_DATA', 'box_top': None, 'box_bottom': None}
    try:
        highs  = pd.to_numeric(df['High'],  errors='coerce').fillna(0).tolist()
        lows   = pd.to_numeric(df['Low'],   errors='coerce').fillna(0).tolist()
        closes = pd.to_numeric(df['Close'], errors='coerce').fillna(0).tolist()
    except KeyError:
        return {'signal': 'INSUFFICIENT_DATA', 'box_top': None, 'box_bottom': None}

    current = closes[-1]
    highs_h, lows_h = highs[:-1], lows[:-1]  # historical only
    n = len(highs_h)

    box_top_idx = box_top = None
    for i in range(n - confirm - 1, -1, -1):
        c = highs_h[i]
        if c == 0: continue
        win = highs_h[i+1:i+1+confirm]
        if len(win) == confirm and all(h < c for h in win):
            box_top_idx, box_top = i, c
            break
    if box_top is None:
        return {'signal': 'NO_BOX', 'box_top': None, 'box_bottom': None}

    seg = lows_h[box_top_idx:]
    box_bottom = None
    for i in range(len(seg) - confirm):
        c = seg[i]
        if c == 0: continue
        win = seg[i+1:i+1+confirm]
        if len(win) == confirm and all(l > c for l in win):
            box_bottom = c; break
    if box_bottom is None:
        valid = [l for l in seg if l > 0]
        box_bottom = min(valid) if valid else None
    if box_bottom is None:
        return {'signal': 'NO_BOX', 'box_top': round(box_top,2), 'box_bottom': None}

    signal = ('BREAKOUT_BUY' if current > box_top else
              'BREAKDOWN_SELL' if current < box_bottom else 'IN_BOX')
    rng = box_top - box_bottom
    return {
        'signal':        signal,
        'box_top':       round(box_top, 2),
        'box_bottom':    round(box_bottom, 2),
        'current_price': round(current, 2),
        'upside_pct':    round((box_top - current)/current*100, 2) if current else 0,
        'pos_in_box':    round((current - box_bottom)/rng*100, 1) if rng else 0,
    }


# ── yfinance helpers ──────────────────────────────────────────────────────────
def _first_df(ticker, *attrs):
    """Return first non-None, non-empty DataFrame from ticker attributes.
    NEVER use Python `or` between DataFrames — raises ambiguous truth-value error."""
    for attr in attrs:
        df = getattr(ticker, attr, None)
        if df is not None and isinstance(df, pd.DataFrame) and not df.empty:
            return df
    return None

def _val(df, *row_names, col=0):
    """Safely get a scalar from a yfinance financial DataFrame."""
    if df is None or df.empty: return None
    for name in row_names:
        if name in df.index:
            try:
                v = df.loc[name].iloc[col]
                return float(v) if pd.notna(v) else None
            except (IndexError, TypeError, ValueError): pass
    return None


# ── Piotroski F-Score (0-9) ───────────────────────────────────────────────────
def run_piotroski(ticker):
    """
    9-point financial-strength score.
    Profitability (4): ROA>0, OCF>0, dROA>0, accruals quality
    Leverage (3):      d-LT-debt, d-current-ratio, no dilution
    Efficiency (2):    d-gross-margin, d-asset-turnover
    Score >=7 STRONG | 4-6 MODERATE | <=3 WEAK
    """
    inc = _first_df(ticker, 'income_stmt', 'financials')
    bal = _first_df(ticker, 'balance_sheet')
    cf  = _first_df(ticker, 'cash_flow', 'cashflow')
    if inc is None or inc.empty:
        return {'f_score': None, 'error': 'no_income_data'}
    s = {}
    ni0=_val(inc,'Net Income',col=0); ni1=_val(inc,'Net Income',col=1)
    a0=_val(bal,'Total Assets',col=0); a1=_val(bal,'Total Assets',col=1)
    roa0=(ni0/a0) if (ni0 and a0) else None
    roa1=(ni1/a1) if (ni1 and a1) else None
    s['F1']=1 if (roa0 and roa0>0) else 0
    ocf0=_val(cf,'Operating Cash Flow','Total Cash From Operating Activities',col=0)
    s['F2']=1 if (ocf0 and ocf0>0) else 0
    s['F3']=1 if (roa0 is not None and roa1 is not None and roa0>roa1) else 0
    s['F4']=1 if (ocf0 and a0 and roa0 is not None and (ocf0/a0)>roa0) else 0
    ltd0=_val(bal,'Long Term Debt',col=0) or 0
    ltd1=_val(bal,'Long Term Debt',col=1) or 0
    lev0=(ltd0/a0) if a0 else None; lev1=(ltd1/a1) if a1 else None
    s['F5']=1 if (lev0 is not None and lev1 is not None and lev0<lev1) else 0
    ca0=_val(bal,'Current Assets','Total Current Assets',col=0)
    cl0=_val(bal,'Current Liabilities','Total Current Liabilities',col=0)
    ca1=_val(bal,'Current Assets','Total Current Assets',col=1)
    cl1=_val(bal,'Current Liabilities','Total Current Liabilities',col=1)
    cr0=(ca0/cl0) if (ca0 and cl0) else None
    cr1=(ca1/cl1) if (ca1 and cl1) else None
    s['F6']=1 if (cr0 is not None and cr1 is not None and cr0>cr1) else 0
    sh0=_val(bal,'Share Issued',col=0); sh1=_val(bal,'Share Issued',col=1)
    s['F7']=(1 if sh0<=sh1 else 0) if (sh0 and sh1) else 1
    rev0=_val(inc,'Total Revenue',col=0); gp0=_val(inc,'Gross Profit',col=0)
    rev1=_val(inc,'Total Revenue',col=1); gp1=_val(inc,'Gross Profit',col=1)
    gm0=(gp0/rev0) if (gp0 and rev0) else None
    gm1=(gp1/rev1) if (gp1 and rev1) else None
    s['F8']=1 if (gm0 is not None and gm1 is not None and gm0>gm1) else 0
    at0=(rev0/a0) if (rev0 and a0) else None
    at1=(rev1/a1) if (rev1 and a1) else None
    s['F9']=1 if (at0 is not None and at1 is not None and at0>at1) else 0
    return {'f_score': sum(s.values()), 'components': s, 'error': None}


# ── Coffee Can Screen (6 criteria, all must pass) ────────────────────────────
def run_coffee_can(ticker):
    """
    US Coffee Can: Revenue CAGR>10%, ROE>15% avg, D/E<1,
    MCap>=$1B, no loss year, Free Cash Flow>0.
    """
    inc = _first_df(ticker, 'income_stmt', 'financials')
    bal = _first_df(ticker, 'balance_sheet')
    cf  = _first_df(ticker, 'cash_flow', 'cashflow')
    info = {}
    try: info = ticker.info or {}
    except: pass
    if inc is None or inc.empty:
        return {'qualifies': False, 'score': '0/6', 'error': 'no_income_data'}

    def series(df, *rows):
        for name in rows:
            if df is not None and name in df.index:
                return [float(v) for v in df.loc[name].dropna() if pd.notna(v)]
        return []

    c = {}
    revs = series(inc, 'Total Revenue')
    if len(revs) >= 2:
        yrs = len(revs)-1
        cagr = ((revs[0]/revs[-1])**(1/yrs)-1)*100 if revs[-1]>0 else None
        c['C1'] = 1 if (cagr and cagr>10) else 0
    else: c['C1'] = 0

    ni_s = series(inc, 'Net Income')
    eq_s = series(bal, 'Stockholders Equity', 'Total Stockholder Equity',
                  'Total Equity Gross Minority Interest')
    roe_l = [ni_s[i]/eq_s[i]*100 for i in range(min(len(ni_s),len(eq_s))) if eq_s[i]>0]
    avg_roe = sum(roe_l)/len(roe_l) if roe_l else None
    c['C2'] = 1 if (avg_roe and avg_roe>15) else 0

    de_raw = info.get('debtToEquity')
    if de_raw is not None:
        de = de_raw/100 if de_raw>10 else de_raw  # normalise percent-form
        c['C3'] = 1 if de<1 else 0
    else:
        ltd_s = series(bal,'Long Term Debt')
        c['C3'] = (1 if (ltd_s and eq_s and eq_s[0]>0 and ltd_s[0]/eq_s[0]<1) else 0)

    mcap = info.get('marketCap')
    if mcap is None:
        try: mcap = ticker.fast_info.market_cap
        except: pass
    c['C4'] = 1 if (mcap and mcap>=1e9) else 0
    c['C5'] = 1 if (ni_s and all(n>0 for n in ni_s)) else 0

    fcf_s = series(cf, 'Free Cash Flow')
    if fcf_s: c['C6'] = 1 if fcf_s[0]>0 else 0
    else:
        ocf_s=series(cf,'Operating Cash Flow','Total Cash From Operating Activities')
        cap_s=series(cf,'Capital Expenditure','Capital Expenditures')
        c['C6'] = 1 if (ocf_s and cap_s and (ocf_s[0]-abs(cap_s[0]))>0) else 0

    total = sum(c.values())
    return {'qualifies': total==COFFEE_CAN_CRITERIA, 'score': f'{total}/{COFFEE_CAN_CRITERIA}',
            'criteria': c, 'roe_avg': round(avg_roe,2) if avg_roe else None, 'error': None}


print('✅ Scan functions loaded')


In [ ]:
# ── Cell 8: SQLite checkpoint helpers ────────────────────────────────────────

def init_db(db_path):
    """Create (or open existing) SQLite database."""
    con = sqlite3.connect(db_path)
    # Use DELETE journal instead of WAL to avoid macOS/Drive I/O errors
    con.execute('PRAGMA journal_mode=DELETE')
    con.execute('''
        CREATE TABLE IF NOT EXISTS results (
            symbol TEXT PRIMARY KEY, name TEXT, exchange TEXT,
            sector TEXT, industry TEXT, market_cap REAL, last_price REAL,
            darvas_signal TEXT, darvas_box_top REAL, darvas_box_bot REAL,
            darvas_current REAL, darvas_upside REAL, darvas_pos REAL,
            f_score INTEGER,
            f1 INT,f2 INT,f3 INT,f4 INT,f5 INT,f6 INT,f7 INT,f8 INT,f9 INT,
            cc_qualifies INTEGER, cc_score TEXT,
            cc_c1 INT,cc_c2 INT,cc_c3 INT,cc_c4 INT,cc_c5 INT,cc_c6 INT,
            cc_roe_avg REAL, scanned_at TEXT, error TEXT)
    ''')
    con.execute('''
        CREATE TABLE IF NOT EXISTS progress (
            symbol TEXT, phase TEXT, done_at TEXT,
            PRIMARY KEY (symbol, phase))
    ''')
    con.commit()
    return con

def already_done(con, phase):
    cur = con.execute('SELECT symbol FROM progress WHERE phase=?', (phase,))
    return {r[0] for r in cur.fetchall()}

def mark_done(con, symbol, phase):
    con.execute('INSERT OR REPLACE INTO progress VALUES (?,?,?)',
                (symbol, phase, datetime.now().isoformat()))

def upsert_result(con, row):
    cols   = ', '.join(row.keys())
    placeh = ', '.join(['?']*len(row))
    update = ', '.join([f'{k}=excluded.{k}' for k in row if k != 'symbol'])
    con.execute(
        f'INSERT INTO results ({cols}) VALUES ({placeh}) '
        f'ON CONFLICT(symbol) DO UPDATE SET {update}',
        list(row.values()))

def bulk_commit(con):
    """Commit accumulated writes in one transaction (efficient + safe)."""
    con.commit()

print('✅ Database helpers loaded')


In [ ]:
# ── Cell 9: Stock universe + Darvas batch runner ─────────────────────────────

def fetch_stock_universe(exchange_filter='all'):
    """
    Download official NASDAQ Trader pipe-delimited stock lists.
    Filters: common stocks only (no ETFs, no test issues, no distressed).
    Returns DataFrame with columns: Symbol, Name, Exchange.
    """
    def _parse(url, is_nasdaq):
        r = requests.get(url, timeout=15)
        r.raise_for_status()
        lines = [l for l in r.text.strip().split('\n')
                 if not l.startswith('File Creation')]
        df = pd.read_csv(io.StringIO('\n'.join(lines)), sep='|', dtype=str).fillna('')
        if is_nasdaq:
            df = df.rename(columns={'Security Name': 'Name'})
            keep = ((df['Test Issue']=='N') & (df['ETF']=='N') &
                    (df['Financial Status']=='N') &
                    df['Symbol'].str.match(r'^[A-Z]{1,5}$', na=False))
            df['Exchange'] = 'NASDAQ'
        else:
            df = df.rename(columns={'ACT Symbol':'Symbol','Security Name':'Name'})
            keep = ((df['Test Issue']=='N') & (df['ETF']=='N') &
                    df['Symbol'].str.match(r'^[A-Z]{1,5}$', na=False))
            df['Exchange'] = df.get('Exchange', pd.Series(['OTHER']*len(df))).map(
                {'N':'NYSE','A':'AMEX','P':'ARCA','Z':'BATS'}).fillna('OTHER')
        return df[keep][['Symbol','Name','Exchange']].copy()

    frames = []
    if exchange_filter in ('all','nasdaq'):
        print('  Fetching NASDAQ stocks...')
        frames.append(_parse(NASDAQ_URL, True))
    if exchange_filter in ('all','nyse','amex','other'):
        print('  Fetching NYSE/AMEX stocks...')
        frames.append(_parse(OTHER_URL, False))
    return pd.concat(frames).drop_duplicates('Symbol').reset_index(drop=True)


def run_darvas_batch(symbols, batch_size=500, pbar=None):
    """
    Batch-download 6-month OHLC for all symbols (500 per yf.download call)
    and compute Darvas Box signals.
    Returns dict: {symbol: darvas_result}.
    """
    results = {}
    batches = [symbols[i:i+batch_size] for i in range(0, len(symbols), batch_size)]
    for batch in batches:
        try:
            raw = yf.download(' '.join(batch), period='6mo', group_by='ticker',
                              auto_adjust=True, threads=True, progress=False)
        except Exception as e:
            log.warning('Batch download failed: %s', e)
            if pbar: pbar.update(len(batch))
            continue
        for sym in batch:
            try:
                df = (raw[sym].dropna() if isinstance(raw.columns, pd.MultiIndex)
                      and sym in raw.columns.get_level_values(0)
                      else raw.dropna() if not isinstance(raw.columns, pd.MultiIndex)
                      else pd.DataFrame())
                results[sym] = _darvas_from_df(df)
            except Exception as e:
                results[sym] = {'signal':'ERROR','box_top':None,'box_bottom':None}
            if pbar: pbar.update(1)
    return results


def scan_fundamentals_one(symbol, name, exchange):
    """
    Fetch yfinance metadata + run Piotroski + Coffee Can for one stock.
    Thread-safe: no shared mutable state.
    """
    row = {'symbol': symbol, 'name': name, 'exchange': exchange,
           'scanned_at': datetime.now().isoformat()}
    try:
        t = yf.Ticker(symbol)
        info = {}
        try: info = t.info or {}
        except: pass
        row['sector']    = info.get('sector', '')
        row['industry']  = info.get('industry', '')
        row['market_cap']= info.get('marketCap')
        try:    row['last_price'] = t.fast_info.last_price
        except: row['last_price'] = info.get('currentPrice') or info.get('regularMarketPrice')

        pio = run_piotroski(t)
        row['f_score'] = pio.get('f_score')
        for i, v in (pio.get('components') or {}).items():
            row[i.lower()] = v   # f1..f9

        cc = run_coffee_can(t)
        row['cc_qualifies'] = 1 if cc.get('qualifies') else 0
        row['cc_score']     = cc.get('score')
        for i, v in (cc.get('criteria') or {}).items():
            row[f'cc_{i.lower()}'] = v   # cc_c1..cc_c6
        row['cc_roe_avg'] = cc.get('roe_avg')
    except Exception as e:
        row['error'] = str(e)[:200]
    return row


print('✅ Universe + batch runner loaded')


In [ ]:
# ── Cell 10: Results compilation ─────────────────────────────────────────────

def compile_outputs(con, today):
    """
    Read SQLite results and write focused CSV files.
    Triple hit = Darvas BREAKOUT_BUY + Piotroski>=7 + Coffee Can qualifies.
    """
    df = pd.read_sql('SELECT * FROM results', con)
    if df.empty:
        print('  No results yet.'); return

    def save(frame, tag, label):
        path = OUTPUT_DIR / f'{tag}_{today}.csv'
        frame.to_csv(path, index=False)
        print(f'  {label:<35} ({len(frame):>5} rows)  →  {path.name}')
        return frame

    save(df, 'scan_summary', 'Full summary')

    bcols = ['symbol','name','exchange','sector','last_price',
             'darvas_signal','darvas_box_top','darvas_box_bot',
             'darvas_current','darvas_upside','darvas_pos']
    save(df[df.darvas_signal=='BREAKOUT_BUY'][[c for c in bcols if c in df.columns]]
         .sort_values('darvas_upside'),
         'darvas_breakouts', 'Darvas BREAKOUT_BUY')

    pcols = ['symbol','name','exchange','sector','last_price','market_cap','f_score',
             'f1','f2','f3','f4','f5','f6','f7','f8','f9']
    save(df[df.f_score.notna() & (df.f_score>=PIOTROSKI_STRONG)]
         [[c for c in pcols if c in df.columns]].sort_values('f_score',ascending=False),
         'strong_piotroski', f'Piotroski F>={PIOTROSKI_STRONG}')

    ccols = ['symbol','name','exchange','sector','industry',
             'last_price','market_cap','cc_score','cc_roe_avg',
             'cc_c1','cc_c2','cc_c3','cc_c4','cc_c5','cc_c6']
    save(df[df.cc_qualifies==1][[c for c in ccols if c in df.columns]]
         .sort_values('market_cap', ascending=False),
         'coffee_can', 'Coffee Can qualifiers')

    triple_mask = ((df.darvas_signal=='BREAKOUT_BUY') &
                   (df.f_score.notna()) & (df.f_score>=PIOTROSKI_STRONG) &
                   (df.cc_qualifies==1))
    tcols = ['symbol','name','exchange','sector','industry','last_price','market_cap',
             'darvas_signal','darvas_box_top','darvas_upside','f_score','cc_score','cc_roe_avg']
    triple = save(df[triple_mask][[c for c in tcols if c in df.columns]]
                  .sort_values('f_score', ascending=False),
                  'triple_hits', '★ Triple hits')

    if not triple.empty:
        print('\n  ★  TRIPLE HIT STOCKS  ★')
        print(f'  {"Symbol":<8} {"Name":<35} {"F":>3}  {"CC":>5}  {"Upside%":>8}')
        print('  ' + '-'*65)
        for _, r in triple.head(20).iterrows():
            print(f'  {r.symbol:<8} {str(r.get("name",""))[:34]:<35} '
                  f'{int(r.f_score):>3}  {str(r.cc_score):>5}  {r.get("darvas_upside",0):>7.1f}%')
    else:
        print('\n  No triple hits found yet (more stocks may still be processing).')


print('✅ Output compiler loaded')


In [ ]:
# ── Cell 11: Main orchestration function ─────────────────────────────────────

def main(exchange=None, limit=None, resume=None, darvas_only=None,
         batch_size=None, workers=None, delay=None):
    """
    Full pipeline. Parameters default to the config cell values.
    Override by passing keyword args: main(limit=200, darvas_only=True)
    """
    # Use config cell values as defaults
    exchange    = exchange    or EXCHANGE
    limit       = limit       if limit is not None else LIMIT
    resume      = resume      if resume is not None else RESUME
    darvas_only = darvas_only if darvas_only is not None else DARVAS_ONLY
    batch_size  = batch_size  or BATCH_SIZE
    workers     = workers     or WORKERS
    delay       = delay       if delay is not None else DELAY
    today       = datetime.today().strftime('%Y%m%d')

    con = init_db(DB_FILE)

    # ── Step 1: Universe ──────────────────────────────────────────────────────
    print('\n── Step 1: Stock Universe ─────────────────────────────────────────')
    universe_file = OUTPUT_DIR / 'stock_universe.csv'
    if universe_file.exists() and resume:
        universe = pd.read_csv(universe_file)
        print(f'  Loaded cached universe: {len(universe):,} stocks')
    else:
        universe = fetch_stock_universe(exchange)
        universe.to_csv(universe_file, index=False)
        print(f'  Fetched {len(universe):,} stocks')
    if limit:
        universe = universe.head(limit)
        print(f'  Limited to {len(universe):,} stocks')

    symbols  = universe['Symbol'].tolist()
    name_map = dict(zip(universe.Symbol, universe.Name))
    exch_map = dict(zip(universe.Symbol, universe.Exchange))

    # ── Step 2: Darvas (batch OHLC) ───────────────────────────────────────────
    print('\n── Phase 1: Darvas Box ───────────────────────────────────────────')
    done_d  = already_done(con, 'darvas') if resume else set()
    todo_d  = [s for s in symbols if s not in done_d]
    print(f'  {len(todo_d):,} to scan ({len(done_d):,} already done)')

    pbar_d = tqdm(total=len(todo_d), unit='stocks', desc='Darvas')
    darvas_results = run_darvas_batch(todo_d, batch_size=batch_size, pbar=pbar_d)
    pbar_d.close()

    for sym, dr in darvas_results.items():
        upsert_result(con, {'symbol':sym,'name':name_map.get(sym,''),
                            'exchange':exch_map.get(sym,''),
                            'darvas_signal':dr.get('signal'),
                            'darvas_box_top':dr.get('box_top'),
                            'darvas_box_bot':dr.get('box_bottom'),
                            'darvas_current':dr.get('current_price'),
                            'darvas_upside':dr.get('upside_pct'),
                            'darvas_pos':dr.get('pos_in_box'),
                            'scanned_at':datetime.now().isoformat()})
        mark_done(con, sym, 'darvas')
    bulk_commit(con)
    sigs = {k: sum(1 for v in darvas_results.values() if v.get('signal')==k)
            for k in ('BREAKOUT_BUY','IN_BOX','BREAKDOWN_SELL')}
    print(f'  Breakouts: {sigs["BREAKOUT_BUY"]:,}  '
          f'In-box: {sigs["IN_BOX"]:,}  '
          f'Breakdown: {sigs["BREAKDOWN_SELL"]:,}')

    if darvas_only:
        print('\n  darvas_only=True — skipping fundamentals.')
        print('\n── Results ──────────────────────────────────────────────────────')
        compile_outputs(con, today)
        con.close(); return

    # ── Step 3: Fundamentals (threaded) ───────────────────────────────────────
    print(f'\n── Phase 2: Piotroski + Coffee Can ({workers} workers) ──────────────')
    done_f = already_done(con, 'fundamentals') if resume else set()
    todo_f = [s for s in symbols if s not in done_f]
    print(f'  {len(todo_f):,} to scan  '
          f'(est. {len(todo_f)*5/workers/60:.0f}–{len(todo_f)*7/workers/60:.0f} min)')

    pbar_f = tqdm(total=len(todo_f), unit='stocks', desc='Fundamentals')

    def _worker(sym):
        r = scan_fundamentals_one(sym, name_map.get(sym,''), exch_map.get(sym,''))
        time.sleep(delay)
        return sym, r

    BATCH_COMMIT = 50  # commit every N stocks for efficiency
    pending = 0
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(_worker, sym): sym for sym in todo_f}
        for future in as_completed(futures):
            sym, row = future.result()
            upsert_result(con, row)
            mark_done(con, sym, 'fundamentals')
            pending += 1
            if pending % BATCH_COMMIT == 0:
                bulk_commit(con)   # batch commit for performance + safety
            pbar_f.update(1)
    bulk_commit(con)  # final commit
    pbar_f.close()

    # ── Step 4: Compile ───────────────────────────────────────────────────────
    print('\n── Results ──────────────────────────────────────────────────────────')
    compile_outputs(con, today)
    con.close()
    print(f'\n  Database: {DB_FILE}')
    print(f'  Outputs:  {OUTPUT_DIR}/')


print('✅ main() loaded and ready')


In [ ]:
# ── Cell 12: 🚀 RUN THE SCREENER ─────────────────────────────────────────────
# Uses settings from Cell 3 (CONFIGURATION).
# You can override individual parameters here, e.g.:
#   main(limit=200, darvas_only=True)   ← quick test
#   main(resume=True)                   ← continue interrupted run
#   main(exchange='nasdaq')             ← NASDAQ only

main()


In [ ]:
# ── Cell 13: Quick view of results ──────────────────────────────────────────
from datetime import datetime
import pandas as pd
today = datetime.today().strftime('%Y%m%d')

# Load the summary CSV
summary_path = OUTPUT_DIR / f'scan_summary_{today}.csv'
triple_path  = OUTPUT_DIR / f'triple_hits_{today}.csv'

if summary_path.exists():
    df = pd.read_csv(summary_path)
    print(f'Total scanned:        {len(df):,}')
    print(f'Darvas BREAKOUT_BUY:  {(df.darvas_signal=="BREAKOUT_BUY").sum():,}')
    pio_ok = df.f_score.notna()
    print(f'Piotroski >=7:        {(pio_ok & (df.f_score>=7)).sum():,}')
    print(f'Coffee Can pass:      {(df.cc_qualifies==1).sum():,}')
    print()
    if triple_path.exists():
        triple = pd.read_csv(triple_path)
        print(f'★ Triple hits: {len(triple)}')
        if not triple.empty:
            display(triple[['symbol','name','sector','last_price',
                             'darvas_upside','f_score','cc_score']].head(20))
else:
    print('No results yet — run Cell 12 first.')
